File 1: SemanticDocumentAnalyzer.py

This is the core logic file.
It:

Reads two documents

Converts text into embeddings using Hugging Face

Compares documents and sections

Produces a clean JSON output

In [ ]:
# SemanticDocumentAnalyzer.py
# -----------------------------------
# This file contains the core logic for comparing two documents semantically.
# It uses a pretrained Hugging Face embedding model to convert text into numbers
# so that a machine can compare meaning instead of raw text.

# Step 1: Install required library
# transformers gives us access to pretrained Hugging Face models
!pip install transformers

# Step 2: Read the two document versions from text files
# We use latin-1 encoding because many policy documents contain special characters
with open("/Policy_v1.txt", "r", encoding="latin-1") as f:
    policy_v1 = f.read()

with open("/Policy_v2.txt", "r", encoding="latin-1") as f:
    policy_v2 = f.read()

# Quick sanity check to confirm both documents loaded
print("Policy V1 length:", len(policy_v1))
print("Policy V2 length:", len(policy_v2))

# Step 3: Load Hugging Face embedding model
# This model converts sentences into dense numeric vectors (embeddings)
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Tokenizer converts text into tokens the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Model generates embeddings from tokens
model = AutoModel.from_pretrained(model_name)

print("Embedding model loaded successfully")

# Step 4: Convert any text into a numeric embedding
# This function is the heart of semantic comparison
def get_embedding(text):
    # Convert text to model-friendly format
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    # Disable gradient calculation because we are only doing inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Average all token vectors to get one vector per document
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding

# Step 5: Generate embeddings for full documents
embedding_v1 = get_embedding(policy_v1)
embedding_v2 = get_embedding(policy_v2)

print("Embedding V1 shape:", embedding_v1.shape)
print("Embedding V2 shape:", embedding_v2.shape)

# Step 6: Compare full documents using cosine similarity
# Cosine similarity measures how close two vectors are in meaning
from torch.nn.functional import cosine_similarity

similarity_score = cosine_similarity(embedding_v1, embedding_v2)
print("Overall similarity score:", similarity_score.item())

# Step 7: Split documents into sections
# This allows us to detect *where* changes happened
import re

def split_into_sections(text):
    # Split based on numbered sections like "1. ", "2. ", etc.
    sections = re.split(r"\n\d+\.\s", text)

    # Remove empty sections and clean text
    sections = [s.strip() for s in sections if s.strip()]
    return sections

sections_v1 = split_into_sections(policy_v1)
sections_v2 = split_into_sections(policy_v2)

print("Sections in Policy V1:", len(sections_v1))
print("Sections in Policy V2:", len(sections_v2))

# Step 8: Compare sections one-by-one
# If similarity drops below threshold, we mark it as a major change
def compare_sections(sections_v1, sections_v2, threshold=0.85):
    major_changes = []

    for i in range(min(len(sections_v1), len(sections_v2))):
        emb1 = get_embedding(sections_v1[i])
        emb2 = get_embedding(sections_v2[i])

        score = cosine_similarity(emb1, emb2).item()

        if score < threshold:
            major_changes.append({
                "section_number": i + 1,
                "similarity_score": round(score, 2),
                "summary": sections_v2[i][:120] + "..."
            })

    return major_changes

major_changes = compare_sections(sections_v1, sections_v2)

# Display detected changes
for change in major_changes:
    print(change)

# Step 9: Format output into clean JSON structure
def format_output(changes):
    return {
        "total_major_changes": len(changes),
        "changes": [
            {
                "section": change["section_number"],
                "summary": change["summary"]
            }
            for change in changes
        ]
    }

final_output = format_output(major_changes)
print(final_output)

# Step 10: Save results to output.json
import json

with open("output.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("Output saved to output.json")


File 2: service.py (BentoML API)

This file turns your logic into a production-style API.
It:

Loads the embedding model once

Accepts two documents as input

Returns similarity and change type

In [ ]:
# service.py
# -----------------------------------
# This file exposes the document comparison logic as an API using BentoML.
# The model is loaded once at startup for efficiency.

import bentoml
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Load the embedding model ONCE
# Loading the model at startup avoids reloading it for every request
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2: Define a BentoML service
@bentoml.service(name="document_change_analyzer")
class DocumentChangeService:

    # Step 3: Define an API endpoint
    @bentoml.api
    def analyze(self, payload: dict) -> dict:
        # Extract documents from request payload
        doc_v1 = payload["doc_v1"]
        doc_v2 = payload["doc_v2"]

        # Convert documents into embeddings
        emb_v1 = model.encode([doc_v1])
        emb_v2 = model.encode([doc_v2])

        # Compare embeddings using cosine similarity
        similarity = cosine_similarity(emb_v1, emb_v2)[0][0]

        # Simple rule to classify change severity
        if similarity > 0.85:
            change_type = "minor_or_no_change"
        else:
            change_type = "major_change"

        # Return structured API response
        return {
            "similarity_score": float(similarity),
            "change_type": change_type
        }